create model:

datasets:https://www.kaggle.com/datasets/akashshingha850/mrl-eye-dataset


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akashshingha850/mrl-eye-dataset")

print("Path to dataset files:", path)

100%|██████████| 329M/329M [00:03<00:00, 91.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/akashshingha850/mrl-eye-dataset/versions/4


In [ ]:
import os

# دیدن اینکه داخل این پوشه چه چیزهایی دانلود شده است
print("محتویات پوشه اصلی:", os.listdir(path))

# فرض کنیم بر اساس ساختار متنی داخل عکس، یک پوشه به نام data وجود دارد
data_path = os.path.join(path, "data")
if os.path.exists(data_path):
    print("محتویات داخل پوشه data:", os.listdir(data_path))

محتویات پوشه اصلی: ['data']
محتویات داخل پوشه data: ['readme.md', 'get_info.py', 'labels.txt', 'val', 'test', 'train', 'split_data.py']


In [ ]:
train_dir=os.path.join(path,"data","train")
val_dir = os.path.join(path, "data", "val")

# ۲. بیاییم فضولی کنیم ببینیم داخل پوشه train چه کلاس‌هایی (پوشه‌هایی) وجود دارد
print("کلاس‌های موجود در پوشه آموزش:", os.listdir(train_dir))

کلاس‌های موجود در پوشه آموزش: ['sleepy', 'awake']


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen=ImageDataGenerator(rescale=1./255)
val_datagen=ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(64,64),
    batch_size=32,
    class_mode="categorical")
validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

Found 50937 images belonging to 2 classes.
Found 16980 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense

model=Sequential()
model.add(Conv2D(32,(3,3),activation='relu',input_shape=(64,64,3)))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(2, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 30752)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,968,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,969,218 (7.51 MB)

 Trainable params: 1,969,218 (7.51 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
history=model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator
)

Epoch 1/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 259s 160ms/step - accuracy: 0.8938 - loss: 0.2652 - val_accuracy: 0.9357 - val_loss: 0.1746
Epoch 2/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 66s 41ms/step - accuracy: 0.9455 - loss: 0.1468 - val_accuracy: 0.9526 - val_loss: 0.1327
Epoch 3/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - accuracy: 0.9567 - loss: 0.1217 - val_accuracy: 0.9627 - val_loss: 0.1081
Epoch 4/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 65s 41ms/step - accuracy: 0.9634 - loss: 0.1051 - val_accuracy: 0.9668 - val_loss: 0.1005
Epoch 5/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 69s 43ms/step - accuracy: 0.9684 - loss: 0.0907 - val_accuracy: 0.9641 - val_loss: 0.1026


In [ ]:
# خط ۱: ذخیره کردن ساختار و وزن‌های مدل در یک فایل واقعی
model.save("drowsiness_real_model.h5")
print("مدل با موفقیت روی هارد موقت کولب ذخیره شد!")
from google.colab import files

# خط ۲: دانلود مستقیم فایل مدل روی کامپیوتر خودت
files.download("drowsiness_real_model.h5")

مدل با موفقیت روی هارد موقت کولب ذخیره شد!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Copying the model file from the Colab temporary drive to your main Google Drive folder.
!cp drowsiness_real_model.h5 /content/drive/MyDrive/

test model:

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf

MODEL_PATH = "drowsiness_real_model.h5"   
model = tf.keras.models.load_model(MODEL_PATH)

INPUT_WIDTH = 64
INPUT_HEIGHT = 64

USE_GRAYSCALE = False  

# ---------------------------------------------------------
# MediaPipe Face Mesh
# ---------------------------------------------------------
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,  
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)


LEFT_EYE_IDX = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDX = [362, 385, 387, 263, 373, 380]


def crop_eye_region(frame, landmarks, eye_idx, frame_w, frame_h, padding=10):
    """
    STEP 2: Crop the eye region out of the full webcam frame.
    Takes the landmark points for one eye, finds a bounding box around them,
    adds a little padding, and returns just that cropped image.
    """
    points = []
    for idx in eye_idx:
        lm = landmarks[idx]
        x, y = int(lm.x * frame_w), int(lm.y * frame_h)
        points.append((x, y))

    points = np.array(points)
    x_min, y_min = points.min(axis=0)
    x_max, y_max = points.max(axis=0)

    # add padding so we don't crop too tightly
    x_min = max(0, x_min - padding)
    y_min = max(0, y_min - padding)
    x_max = min(frame_w, x_max + padding)
    y_max = min(frame_h, y_max + padding)

    eye_crop = frame[y_min:y_max, x_min:x_max]
    return eye_crop


def preprocess_eye(eye_img):


    if eye_img.size == 0:
        return None

    if USE_GRAYSCALE:
        eye_img = cv2.cvtColor(eye_img, cv2.COLOR_BGR2GRAY)
    else:
       
        eye_img = cv2.cvtColor(eye_img, cv2.COLOR_BGR2RGB)

    eye_img = cv2.resize(eye_img, (INPUT_WIDTH, INPUT_HEIGHT))
    eye_img = eye_img.astype("float32") / 255.0  # normalize 0-1

    if USE_GRAYSCALE:
        eye_img = np.expand_dims(eye_img, axis=-1)  # add channel dim -> (H, W, 1)

    eye_img = np.expand_dims(eye_img, axis=0)  # add batch dim -> (1, H, W, C)
    return eye_img


def predict_eye_state(eye_img):
 
    pred = model.predict(eye_img, verbose=0)

   
    predicted_class = np.argmax(pred[0])

    
    closed = predicted_class == 0
    return closed


# ---------------------------------------------------------

CLOSED_FRAMES_THRESHOLD = 20

closed_counter = 0

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    frame_h, frame_w = frame.shape[:2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = face_mesh.process(rgb_frame)

    status_text = "No face detected"
    both_eyes_closed = False

    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark

        left_eye_crop = crop_eye_region(frame, landmarks, LEFT_EYE_IDX, frame_w, frame_h)
        right_eye_crop = crop_eye_region(frame, landmarks, RIGHT_EYE_IDX, frame_w, frame_h)

        left_input = preprocess_eye(left_eye_crop)
        right_input = preprocess_eye(right_eye_crop)

        if left_input is not None and right_input is not None:
            left_closed = predict_eye_state(left_input)
            right_closed = predict_eye_state(right_input)
            both_eyes_closed = left_closed and right_closed

            status_text = "EYES CLOSED" if both_eyes_closed else "Eyes open"

    if both_eyes_closed:
        closed_counter += 1
    else:
        closed_counter = 0

    if closed_counter >= CLOSED_FRAMES_THRESHOLD:
        status_text = "DROWSY ALERT!"
        # TODO: this is where you'd call Gemini API / send Bale message
        # e.g. send_bale_alert() or call_gemini_for_message()

    cv2.putText(frame, status_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 0, 255), 2)
    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()